# 08 - Performance e manutenção

## Objetivo

Demonstrar inspeção e manutenção de arquivos/snapshots.

## Valor para a PRODEMGE

Cargas incrementais podem gerar arquivos pequenos e muitos snapshots. Esta etapa mostra como manter a tabela saudável.

## Como usar

1. Substitua os placeholders (`<datahub_link>`, `<usuário>` e `<senha>`) no primeiro bloco de código.
2. Execute a célula de configuração Livy.
3. Execute as células da demo.
4. No final, encerre a sessão Livy.

> Este notebook não executa Spark localmente. Todo código Spark SQL/PySpark é submetido ao Livy3 via REST API.

In [ ]:
import requests
import time
import json
import urllib3

# Remove warnings de certificado SSL self-signed
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# =====================================================================
# CONFIGURAÇÕES DO LIVY3
# =====================================================================

LIVY_URL = "<datahub_link>"
USERNAME = "<usuário>"
PASSWORD = "<senha>"


BASE_LIVY_URL = LIVY_URL.rstrip("/")

# =====================================================================
# CONFIGURAÇÕES SPARK
# =====================================================================

SESSION_CONF = {
    "spark.app.name": "PRODEMGE_Iceberg_Demo_Livy3",

    "spark.executor.memory": "4g",
    "spark.executor.cores": "2",
    "spark.executor.instances": "2",

    "spark.driver.memory": "2g",

    "spark.sql.catalog.iceberg_prod":
        "org.apache.iceberg.spark.SparkCatalog",

    "spark.sql.catalog.iceberg_prod.type":
        "hive",

    "spark.sql.extensions":
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
}

# =====================================================================
# CLIENTE HTTP
# =====================================================================

http = requests.Session()

http.auth = (USERNAME, PASSWORD)
http.verify = False

HEADERS = {
    "Content-Type": "application/json"
}

# =====================================================================
# AUXILIARES
# =====================================================================

def pretty_json(obj):
    print(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False
        )
    )

# =====================================================================
# CRIAÇÃO DA SESSÃO LIVY
# =====================================================================

def create_livy_session(
    kind="pyspark",
    conf=None,
    timeout_seconds=600
):

    payload = {
        "kind": kind,
        "conf": conf or SESSION_CONF
    }

    response = http.post(
        f"{BASE_LIVY_URL}/sessions",
        headers=HEADERS,
        data=json.dumps(payload),
        verify=False
    )

    response.raise_for_status()

    session_id = response.json()["id"]

    print(f"Sessão Livy criada: {session_id}")

    start = time.time()

    while True:

        response = http.get(
            f"{BASE_LIVY_URL}/sessions/{session_id}",
            verify=False
        )

        response.raise_for_status()

        payload = response.json()

        state = payload.get("state")

        print(f"Estado da sessão: {state}")

        if state == "idle":

            print(
                "Sessão Livy pronta para receber statements."
            )

            return session_id

        if state in ["dead", "error", "killed"]:

            pretty_json(payload)

            raise RuntimeError(
                f"Falha ao criar sessão Livy. Estado: {state}"
            )

        if time.time() - start > timeout_seconds:

            raise TimeoutError(
                "Timeout aguardando sessão Livy ficar idle."
            )

        time.sleep(5)

# =====================================================================
# EXECUÇÃO DE STATEMENTS
# =====================================================================

def submit_statement(
    session_id,
    code,
    kind="pyspark",
    timeout_seconds=900
):

    payload = {
        "code": code,
        "kind": kind
    }

    response = http.post(
        f"{BASE_LIVY_URL}/sessions/{session_id}/statements",
        headers=HEADERS,
        data=json.dumps(payload),
        verify=False
    )

    response.raise_for_status()

    statement_id = response.json()["id"]

    print(
        f"Statement submetido: {statement_id}"
    )

    start = time.time()

    while True:

        response = http.get(
            f"{BASE_LIVY_URL}/sessions/{session_id}/statements/{statement_id}",
            verify=False
        )

        response.raise_for_status()

        result = response.json()

        state = result.get("state")

        print(
            f"Estado do statement: {state}"
        )

        if state == "available":

            output = result.get(
                "output",
                {}
            )

            pretty_json(output)

            return output

        if state in [
            "error",
            "cancelling",
            "cancelled"
        ]:

            pretty_json(result)

            raise RuntimeError(
                f"Statement falhou. Estado: {state}"
            )

        if time.time() - start > timeout_seconds:

            raise TimeoutError(
                "Timeout aguardando statement finalizar."
            )

        time.sleep(3)

# =====================================================================
# ENCERRAMENTO DA SESSÃO
# =====================================================================

def close_livy_session(session_id):

    response = http.delete(
        f"{BASE_LIVY_URL}/sessions/{session_id}",
        verify=False
    )

    if response.status_code in [
        200,
        202,
        204
    ]:

        print(
            f"Sessão Livy encerrada: {session_id}"
        )

    else:

        print(
            f"Não foi possível encerrar a sessão {session_id}"
        )

        print(response.text)

# =====================================================================
# TESTE
# =====================================================================

livy_session_id = create_livy_session()

print(
    f"Sessão criada com sucesso: {livy_session_id}"
)

In [ ]:
# Normaliza o username para usar no nome da tabela sem caracteres inválidos.
table_suffix = "".join(ch if ch.isalnum() or ch == "_" else "_" for ch in USERNAME).strip("_")
table_name = f"beneficios_{table_suffix}"
particionado_table = f"beneficios_particionados_{table_suffix}"
staging_table = f"beneficios_staging_{table_suffix}"
csv_table = f"beneficios_csv_{table_suffix}"

spark_code = f"""
# =====================================================================
# PERFORMANCE E MANUTENÇÃO
# =====================================================================

print("Arquivos atuais da tabela:")
spark.sql(\"\"\"
SELECT
    COUNT(*) AS quantidade_arquivos,
    SUM(record_count) AS quantidade_registros,
    SUM(file_size_in_bytes) AS tamanho_total_bytes
FROM governo_mg.{table_name}.files
\"\"\").show(truncate=False)

# Compactação/rewrite de arquivos.
# Em tabelas pequenas, pode não haver muito a compactar, mas o comando demonstra o recurso.
print("Executando rewrite_data_files...")
spark.sql(\"\"\"
CALL system.rewrite_data_files(
    table => 'governo_mg.{table_name}'
)
\"\"\").show(truncate=False)

print("Snapshots atuais:")
spark.sql(\"\"\"
SELECT snapshot_id, committed_at, operation
FROM governo_mg.{table_name}.snapshots
ORDER BY committed_at DESC
\"\"\").show(truncate=False)

# ATENÇÃO:
# expire_snapshots remove snapshots antigos e pode impactar time travel.
# Para demo, deixamos comentado por segurança.
print(\"\"\"
Para expirar snapshots antigos, use com cautela:

CALL system.expire_snapshots(
    table => 'governo_mg.{table_name}',
    older_than => TIMESTAMP '2025-01-01 00:00:00'
)
\"\"\")

print("Manutenção demonstrada com sucesso.")
"""

submit_statement(livy_session_id, spark_code)

In [ ]:
# Encerre a sessão ao final do notebook.
close_livy_session(livy_session_id)